# Portfolio Optimiser Demo

This notebook demonstrates the key features of the Portfolio Optimiser library:

1. **Data Loading** - Load and analyze stock price data
2. **Efficient Frontier** - Construct the mean-variance efficient frontier
3. **Portfolio Strategies** - Compare Max Sharpe, Min Variance, Equal Weight, and Risk Parity
4. **Constraints** - Apply position limits and sector constraints
5. **Backtesting** - Walk-forward backtest with transaction costs
6. **Risk Attribution** - Analyze portfolio risk contributions

In [ ]:
# Import the library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portfolio_optimiser import (
    # Data loading
    load_prices,
    calculate_returns,
    annualize_returns,
    calculate_covariance,
    calculate_correlation,
    calculate_annual_volatility,
    # Optimization
    efficient_frontier,
    max_sharpe_ratio,
    min_variance,
    equal_weight,
    risk_parity,
    portfolio_return,
    portfolio_volatility,
    compare_strategies,
    # Constraints
    PortfolioConstraints,
    # Costs
    TransactionCostModel,
    # Backtesting
    PortfolioBacktester,
    # Reporting
    print_portfolio_summary,
    plot_risk_pie,
    plot_correlation_heatmap,
)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load and Explore Data

In [ ]:
# Load price data
prices = load_prices("data/sample_price_data.csv")

# Select 5 tech stocks for analysis
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
prices = prices[tickers]

print(f"Loaded {len(prices)} days of price data")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")
prices.head()

In [ ]:
# Calculate returns and statistics
returns = calculate_returns(prices)
expected_returns = annualize_returns(returns).values
cov_matrix = calculate_covariance(returns).values
corr_matrix = calculate_correlation(returns).values
volatilities = calculate_annual_volatility(returns).values

# Display summary statistics
stats_df = pd.DataFrame({
    'Annual Return': expected_returns,
    'Annual Volatility': volatilities,
    'Sharpe (Rf=2%)': (expected_returns - 0.02) / volatilities
}, index=tickers)

print("\n=== Stock Statistics ===")
stats_df.round(4)

In [ ]:
# Plot correlation heatmap
plot_correlation_heatmap(corr_matrix, tickers)

## 2. Efficient Frontier

In [ ]:
# Construct the efficient frontier
frontier_vols, frontier_returns, frontier_weights = efficient_frontier(
    expected_returns,
    cov_matrix,
    n_points=50,
)

# Plot the frontier with individual stocks
plt.figure(figsize=(10, 6))
plt.plot(frontier_vols, frontier_returns, 'b-', linewidth=2, label='Efficient Frontier')
plt.scatter(volatilities, expected_returns, s=100, c='red', marker='o', label='Individual Stocks')

for i, ticker in enumerate(tickers):
    plt.annotate(ticker, (volatilities[i], expected_returns[i]), 
                 xytext=(5, 5), textcoords='offset points')

plt.xlabel('Risk (Volatility)')
plt.ylabel('Expected Return')
plt.title('Efficient Frontier')
plt.legend()
plt.grid(True)
plt.show()

## 3. Compare Portfolio Strategies

In [ ]:
# Compare all strategies
strategies, strategy_points = compare_strategies(
    expected_returns,
    cov_matrix,
    tickers,
    risk_free_rate=0.02,
)

In [ ]:
# Visualize strategies on the frontier
plt.figure(figsize=(10, 6))
plt.plot(frontier_vols, frontier_returns, 'b-', linewidth=2, label='Efficient Frontier')

colors = ['green', 'orange', 'purple', 'red']
for i, (name, (vol, ret)) in enumerate(strategy_points.items()):
    plt.scatter(vol, ret, s=150, c=colors[i], marker='*', label=name, zorder=5)
    plt.annotate(name, (vol, ret), xytext=(8, -5), textcoords='offset points', fontsize=9)

plt.xlabel('Risk (Volatility)')
plt.ylabel('Expected Return')
plt.title('Portfolio Strategies on Efficient Frontier')
plt.legend()
plt.grid(True)
plt.show()

## 4. Constrained Optimization

In [ ]:
# Create constraints: max 30% per asset
constraints = PortfolioConstraints(max_weight=0.30, min_weight=0.0)

# Unconstrained Max Sharpe
unconstrained_weights = max_sharpe_ratio(expected_returns, cov_matrix, risk_free_rate=0.02)

# Constrained Max Sharpe
from portfolio_optimiser.optimizer import maximize_sharpe
constrained_result = maximize_sharpe(
    expected_returns, cov_matrix, len(tickers),
    constraints_obj=constraints, risk_free_rate=0.02
)
constrained_weights = constrained_result.x

# Compare
comparison_df = pd.DataFrame({
    'Unconstrained': unconstrained_weights,
    'Constrained (max 30%)': constrained_weights,
}, index=tickers)

print("=== Weight Comparison ===")
print(comparison_df.round(4))
print(f"\nUnconstrained Sharpe: {(portfolio_return(unconstrained_weights, expected_returns) - 0.02) / portfolio_volatility(unconstrained_weights, cov_matrix):.4f}")
print(f"Constrained Sharpe:   {(portfolio_return(constrained_weights, expected_returns) - 0.02) / portfolio_volatility(constrained_weights, cov_matrix):.4f}")

In [ ]:
# Visualize weight comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(tickers, unconstrained_weights)
axes[0].axhline(y=0.30, color='r', linestyle='--', label='30% limit')
axes[0].set_title('Unconstrained Max Sharpe')
axes[0].set_ylabel('Weight')
axes[0].legend()

axes[1].bar(tickers, constrained_weights)
axes[1].axhline(y=0.30, color='r', linestyle='--', label='30% limit')
axes[1].set_title('Constrained Max Sharpe (max 30%)')
axes[1].set_ylabel('Weight')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Backtesting with Transaction Costs

In [ ]:
# Set up transaction cost model
cost_model = TransactionCostModel(
    commission_rate=0.001,   # 0.1% commission
    spread_cost=0.0005,      # 0.05% spread
    min_commission=1.0,      # $1 minimum
)

# Create optimizer wrapper
def optimizer_wrapper(exp_ret, cov_mat):
    return max_sharpe_ratio(exp_ret, cov_mat, risk_free_rate=0.02)

# Run backtest
backtester = PortfolioBacktester(
    optimizer_func=optimizer_wrapper,
    rebalance_months=1,        # Monthly rebalancing
    cost_model=cost_model,
    starting_value=100000,     # $100k initial capital
    risk_free_rate=0.02,
    lookback_days=60,          # 60-day trailing window
)

backtester.run(prices)
backtester.calculate_metrics()

In [ ]:
# Print metrics
backtester.print_metrics()

In [ ]:
# Plot backtest results
backtester.plot_results(title="Max Sharpe Monthly Rebalancing Backtest")

## 6. Risk Attribution

In [ ]:
# Analyze risk contributions for Max Sharpe portfolio
weights = max_sharpe_ratio(expected_returns, cov_matrix, risk_free_rate=0.02)

report = print_portfolio_summary(
    weights,
    expected_returns,
    cov_matrix,
    tickers,
)

In [ ]:
# Plot risk contribution pie chart
plot_risk_pie(tickers, report['pct_risk_contributions'])

## Key Takeaways

1. **Concentration Risk**: Unconstrained optimization often leads to extreme allocations (90%+ in single assets)

2. **Diversification Ratio > 1**: The portfolio benefits from diversification due to imperfect correlations

3. **Transaction Costs Matter**: Even small costs compound over time with frequent rebalancing

4. **Risk Parity**: Provides more balanced risk exposure across assets compared to traditional optimization